# Model A — daily Daymet pilot

Shared LSTM predicting daily discharge at the catchment outlet using whole-basin inputs.

**Pilot:** 10 catchments (5 Canada, 5 US), across headwater, meso-scale and macro-scale groups. This deterministic coverage-screened sample is for checking the workflow, not a representative research sample.

**Dynamic features (7):** precipitation, maximum/minimum temperature, shortwave radiation, vapour pressure, day length, potential evapotranspiration.

**Static features (5):** basin area, mean elevation, mean slope, stream density, elongation ratio. This initial topographic baseline can later be expanded with documented soil/vegetation descriptors before the final experiment is frozen.

**Task:** same-day rainfall–runoff simulation (target-day weather included), not a forecast with unknown future weather.

**Exploratory split:** train 1980–1999; validate 2000–2004; test 2005–2009. Lookback 365 days; the first 364 training days supply context. Prior-period weather supplies validation/test context. Targets are discharge in mm/day, converted from m3/s with delineated basin area.

Partial-day and negative discharge are excluded. Estimated and ice-affected observations are retained for this pilot; final quality screening needs a sensitivity check. Missing-weather windows are excluded. No discharge observations are used as inputs. Normalization uses training periods only.


In [1]:
from pathlib import Path
import json, csv, subprocess
BASE = Path.home() / 'Desktop/LSTM_RR_Modelling/Model_A'
PYTHON = BASE / '.venv/bin/python'
config = json.loads((BASE / 'pilot_config.json').read_text())
print('Weather:', config['forcing'])
print('Inputs:', config['features'], config['static'])
for b in config['basins']:
    print(b['basin_id'], b['group'], 'train/validation/test windows:', b['windows'])


Weather: daymet
Inputs: ['prcp', 'tmax', 'tmin', 'srad', 'vp', 'dayl', 'pet'] ['basin_area', 'elev_mean', 'slope_mean', 'stream_density', 'elongation_ratio']
CAN_01AD003 macro-scale train/validation/test windows: [6941, 1827, 1826]
CAN_01AF007 meso-scale train/validation/test windows: [6941, 1827, 1826]
CAN_01AF009 meso-scale train/validation/test windows: [2923, 1827, 1826]
CAN_01AJ003 macro-scale train/validation/test windows: [6941, 1827, 1826]
CAN_01BJ012 headwater train/validation/test windows: [5480, 1827, 1826]
USA_01013500 macro-scale train/validation/test windows: [6941, 1827, 1826]
USA_01022500 meso-scale train/validation/test windows: [6941, 1827, 1826]
USA_01031500 meso-scale train/validation/test windows: [6941, 1827, 1826]
USA_01073000 headwater train/validation/test windows: [6941, 1827, 1826]
USA_01121000 headwater train/validation/test windows: [6941, 1827, 1826]


## Check the prepared data and model
The smoke test performs a forward/backward pass and checks that validation/test targets cannot affect preprocessing.

In [4]:
result = subprocess.run([str(PYTHON), str(BASE/'train_model_a.py'), '--smoke'],
                        capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()


SMOKE PASS cpu loss 0.31692370772361755 windows {'train': 63931, 'validation': 18270, 'test': 18260}

/home/pollen26/Desktop/LSTM_RR_Modelling/Model_A/train_model_a.py:87: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /tmp/build_wheels_tmp.103000/python-3.11/torch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  print('SMOKE PASS',device,'loss',float(loss),'windows',{s:len(d) for s,d in ds.items()},flush=True)



## Train the pilot

Run the following cell to train. It starts a 20-epoch diagnostic run, sampling 200 batches per epoch with approximately equal basin probability. It selects the best checkpoint using validation loss, then evaluates the test period once. The model has one LSTM layer with 64 hidden units and a linear output. Loss is squared error normalized by each basin's training discharge variability.

A pilot run is not a tuned scientific benchmark. Freeze features, sampling, splits and tuning budgets before comparing heterogeneity models. Multiple seeds are necessary for the final study.


In [ ]:
subprocess.run([str(PYTHON), '-u', str(BASE/'train_model_a.py'),
                '--epochs', '20', '--steps', '200', '--seed', '42'], check=True)


## Inspect saved results
Each timestamped run contains the checkpoint, training-only scalers, configuration, history, per-basin test predictions and NSE/KGE/bias metrics. Predictions are not clipped; negative predicted flows remain visible for diagnosis.

In [ ]:
runs = sorted((BASE/'runs').glob('*')) if (BASE/'runs').exists() else []
if runs:
    latest = runs[-1]
    print('Latest run:', latest)
    if (latest/'metrics.csv').exists():
        for row in csv.DictReader((latest/'metrics.csv').open()): print(row)
    elif (latest/'history.json').exists():
        print(json.loads((latest/'history.json').read_text())[-1])
else:
    print('No training run yet. Data preparation and smoke testing are separate from training.')


## Implementation
The two source files below are kept alongside the notebook for review and reuse. Preparation requires the Nibi netcdf module; training uses the isolated .venv.

In [ ]:
print((BASE/'prepare_daymet.py').read_text())

In [ ]:
print((BASE/'train_model_a.py').read_text())